**[Course Home](../README.md) | [Python Lab](../python/04_markov_decision_processes_and_bellman_optimality.ipynb) | [Prev: Ch 03: Greedy Bounds & CELF](03_nemhauser_wolsey_greedy_guarantees.ipynb) | [Next: Ch 05: POMDPs & Belief States](05_pomdps_and_belief_state_mdps.ipynb)**

*Part II: Sequential Control & Dynamic Programming — Chapter 04 of 11*

---

# Chapter 04: Markov Decision Processes — Bellman Equations, Value/Policy Iteration & Sequential Stochastic Control
### *From Static Submodular Portfolios to Dynamic Feedback: Contraction Mappings, Dynamic Programming Operators, and Optimal Fleet Maintenance*

---

## 1. From Static Combinatorial Selection to Sequential Dynamic Control

In Chapters 01 through 03, we analyzed static combinatorial optimization. In that setting, an engineering team selects a subset of observability probes, test suites, or canary assertions subject to a static budget constraint:

$$
\max_{S \subseteq \mathcal{V}} f(S) \quad \text{subject to} \quad c(S) \le B
$$

When the objective function $f(S)$ exhibits submodularity (diminishing marginal returns), greedy algorithms with lazy evaluations (CELF) provide provable $(1 - 1/e) \approx 63.2\%$ approximation guarantees.

However, static portfolio selection rests on an assumption that often fails in production systems: **the underlying system state is invariant to the decisions made**. Static models assume that selecting a test suite or deploying a monitoring probe does not alter the physical degradation, queue dynamics, memory fragmentation, or operational health of the infrastructure over time.

### The Dynamic Sequential Control Reality

In production systems—such as continuous integration (CI) worker fleets, automated hardware testbeds, distributed cloud microservices, and database transaction engines—decisions do not occur in an isolated vacuum. Every operational action exerts a **dual effect**:
1. **Immediate Payoff:** It yields an instantaneous reward or incurs an instantaneous cost (e.g., executing a build job yields developer throughput, while running a comprehensive diagnostics pass costs CPU cycles and execution delay).
2. **State Transition:** It alters the probability distribution over future physical system states (e.g., running a high-stress test on an already degraded runner increases the probability of an out-of-memory kernel panic, while taking the node offline for container garbage collection and cache flushing resets it to a pristine state).

This feedback loop creates an ongoing tension between **short-term myopic exploitation** and **long-term state health preservation**. A controller that greedily maximizes immediate throughput will run degraded worker nodes until they crash catastrophically, blocking merge queues and requiring expensive manual human interventions. Conversely, an overly defensive controller that reboots worker nodes too frequently sacrifices computational capacity.

To solve this problem rigorously, we must transition from static subset selection to **stochastic sequential decision processes**, where an automated controller observes the current state of the system, selects an action, receives an immediate reward, and influences the stochastic trajectory of future states over an infinite horizon.


## 2. Mathematical Formulation of Markov Decision Processes

A Markov Decision Process (MDP) provides the mathematical framework for modeling sequential decision-making under uncertainty where outcomes are partly random and partly under the control of a decision-maker.

Formally, a discrete-time, infinite-horizon discounted Markov Decision Process is defined as a 5-tuple:

$$
\mathcal{M} = \langle \mathcal{S}, \mathcal{A}, \mathcal{P}, \mathcal{R}, \gamma \rangle
$$

where:
- $\mathcal{S}$ is the **state space**, a finite set containing all valid configurations of the environment ($|\mathcal{S}| < \infty$).
- $\mathcal{A}$ is the **action space**, a finite set containing all control actions available to the agent ($|\mathcal{A}| < \infty$).
- $\mathcal{P}: \mathcal{S} \times \mathcal{A} \times \mathcal{S} \to [0, 1]$ is the **transition probability tensor**:

$$
\mathcal{P}(s' \mid s, a) = \mathbb{P}(S_{t+1} = s' \mid S_t = s, A_t = a)
$$

satisfying the row-stochasticity condition $\sum_{s' \in \mathcal{S}} \mathcal{P}(s' \mid s, a) = 1, \forall s \in \mathcal{S}, a \in \mathcal{A}$.
- $\mathcal{R}: \mathcal{S} \times \mathcal{A} \to \mathbb{R}$ is the **expected reward function**, defined as:

$$
\mathcal{R}(s, a) = \mathbb{E}[R_{t+1} \mid S_t = s, A_t = a]
$$

- $\gamma \in [0, 1)$ is the **discount factor**, which models the time-preference of rewards, guarantees mathematical convergence over infinite horizons, and accounts for the probability $(1 - \gamma)$ of process termination at each discrete step.

### The Markov Property: Definition and Systems Significance

The defining axiom of an MDP is the **Markov property** (first-order history independence). A stochastic sequence of states and rewards satisfies the Markov property if the conditional probability distribution of future states depends solely upon the present state and action, and is conditionally independent of all preceding history:

$$
\mathbb{P}(S_{t+1} = s', R_{t+1} = r \mid S_t = s_t, A_t = a_t, S_{t-1} = s_{t-1}, A_{t-1} = a_{t-1}, \dots, S_0 = s_0, A_0 = a_0) = \mathbb{P}(S_{t+1} = s', R_{t+1} = r \mid S_t = s_t, A_t = a_t)
$$

#### Systems Implication: State Sufficiency

In software and systems engineering, the Markov property establishes the requirement for **state sufficiency**. 

If the probability that a cloud worker node crashes during a test run depends not only on its current memory utilization $m_t$, but also on the number of consecutive failed jobs it has processed over the past two hours, then the scalar $m_t$ alone **violates the Markov property**. 

To restore the Markov property, the engineer must augment the state representation:

$$
s_t = \begin{bmatrix} \text{Memory Utilization (\%)} \\ \text{Disk Cache Fragmentation (\%)} \\ \text{Consecutive Job Failures} \\ \text{Time Since Last Reboot (seconds)} \end{bmatrix}
$$

A state $s_t$ is sufficient if and only if it encapsulates all historical information required to make the probability distribution of future trajectories conditionally independent of the past.


## 3. Policies, Returns, and Value Functions

A **policy** defines the decision-making rule of the controller:
- A **stationary deterministic policy** is a mapping $\pi: \mathcal{S} \to \mathcal{A}$ that assigns a specific action $a = \pi(s)$ to each state $s$.
- A **stationary stochastic policy** is a mapping $\pi: \mathcal{S} \times \mathcal{A} \to [0, 1]$, where $\pi(a \mid s) = \mathbb{P}(A_t = a \mid S_t = s)$, with $\sum_{a \in \mathcal{A}} \pi(a \mid s) = 1$.

Stationarity implies that the policy depends only on the current state $s$, not on the absolute wall-clock time $t$.

### Cumulative Discounted Return

Given a policy $\pi$, the system generates a trajectory $S_0, A_0, R_1, S_1, A_1, R_2, \dots$. The cumulative discounted return $G_t$ from time step $t$ onward is:

$$
G_t \equiv \sum_{k=0}^\infty \gamma^k R_{t+k+1} = R_{t+1} + \gamma R_{t+2} + \gamma^2 R_{t+3} + \dots
$$

The discount factor $\gamma \in [0, 1)$ ensures mathematical convergence for bounded rewards ($|G_t| \le \frac{R_{\max}}{1 - \gamma} < \infty$) and establishes an **effective planning horizon** $H_{\text{eff}} \approx \frac{1}{1 - \gamma}$. For $\gamma = 0.9$, $H_{\text{eff}} \approx 10$ steps; for $\gamma = 0.99$, $H_{\text{eff}} \approx 100$ steps.

### State-Value and Action-Value Functions

We quantify the long-term desirability of states and state-action pairs using two functions:
1. **The State-Value Function $V^\pi(s)$:** The expected cumulative discounted return starting from state $s$ and following policy $\pi$ thereafter:

$$
V^\pi(s) \equiv \mathbb{E}_\pi [G_t \mid S_t = s] = \mathbb{E}_\pi \left[ \sum_{k=0}^\infty \gamma^k R_{t+k+1} \;\middle|\; S_t = s \right]
$$

2. **The Action-Value Function $Q^\pi(s, a)$:** The expected cumulative discounted return starting from state $s$, executing action $a$, and subsequently adhering to policy $\pi$:

$$
Q^\pi(s, a) \equiv \mathbb{E}_\pi [G_t \mid S_t = s, A_t = a] = \mathbb{E}_\pi \left[ \sum_{k=0}^\infty \gamma^k R_{t+k+1} \;\middle|\; S_t = s, A_t = a \right]
$$


## 4. The Bellman Equations: Expectation and Optimality

The fundamental insight of dynamic programming is that value functions satisfy recursive relationships linking the value of the current state to the expected values of successor states.

### The Bellman Expectation Equation

By unrolling the cumulative return $G_t = R_{t+1} + \gamma G_{t+1}$ and invoking the Law of Total Expectation, we derive the **Bellman Expectation Equation** for $V^\pi(s)$:

$$
V^\pi(s) = \sum_{a \in \mathcal{A}} \pi(a \mid s) \left[ \mathcal{R}(s, a) + \gamma \sum_{s' \in \mathcal{S}} \mathcal{P}(s' \mid s, a) V^\pi(s') \right]
$$

and for the action-value function:

$$
Q^\pi(s, a) = \mathcal{R}(s, a) + \gamma \sum_{s' \in \mathcal{S}} \mathcal{P}(s' \mid s, a) \sum_{a' \in \mathcal{A}} \pi(a' \mid s') Q^\pi(s', a')
$$

#### Matrix Formulation of Policy Evaluation

For a deterministic policy $\pi$, let $\mathbf{V}^\pi \in \mathbb{R}^{|\mathcal{S}|}$ be the state-value vector, $\mathbf{R}_\pi \in \mathbb{R}^{|\mathcal{S}|}$ be the reward vector with entries $\mathcal{R}(s, \pi(s))$, and $\mathbf{P}_\pi \in \mathbb{R}^{|\mathcal{S}| \times |\mathcal{S}|}$ be the transition matrix with entries $\mathcal{P}(s' \mid s, \pi(s))$. The Bellman expectation equation becomes:

$$
\mathbf{V}^\pi = \mathbf{R}_\pi + \gamma \mathbf{P}_\pi \mathbf{V}^\pi \implies (I - \gamma \mathbf{P}_\pi) \mathbf{V}^\pi = \mathbf{R}_\pi
$$

Because $\mathbf{P}_\pi$ is row-stochastic, its spectral radius satisfies $\rho(\mathbf{P}_\pi) = 1$. Since $\gamma < 1$, the matrix $\gamma \mathbf{P}_\pi$ has spectral radius $\rho(\gamma \mathbf{P}_\pi) = \gamma < 1$. Thus, $(I - \gamma \mathbf{P}_\pi)$ is strictly invertible, and its inverse is given by the absolutely convergent Neumann series $\sum_{k=0}^\infty (\gamma \mathbf{P}_\pi)^k$. The exact value vector can be solved directly in closed form:

$$
\mathbf{V}^\pi = (I - \gamma \mathbf{P}_\pi)^{-1} \mathbf{R}_\pi
$$

### The Bellman Optimality Equation

The fundamental goal in an MDP is to find an **optimal policy** $\pi^*$ that achieves the maximum possible expected return from every state simultaneously: $V^*(s) \equiv \max_\pi V^\pi(s)$ for all $s \in \mathcal{S}$. 

Bellman established that the optimal value function satisfies the non-linear **Bellman Optimality Equation**:

$$
V^*(s) = \max_{a \in \mathcal{A}} \left[ \mathcal{R}(s, a) + \gamma \sum_{s' \in \mathcal{S}} \mathcal{P}(s' \mid s, a) V^*(s') \right]
$$

and for the optimal action-value function:

$$
Q^*(s, a) = \mathcal{R}(s, a) + \gamma \sum_{s' \in \mathcal{S}} \mathcal{P}(s' \mid s, a) \max_{a' \in \mathcal{A}} Q^*(s', a')
$$

with $V^*(s) = \max_{a \in \mathcal{A}} Q^*(s, a)$. Once $V^*$ is known, an optimal policy $\pi^*$ is extracted greedily:

$$
\pi^*(s) = \arg\max_{a \in \mathcal{A}} \left[ \mathcal{R}(s, a) + \gamma \sum_{s' \in \mathcal{S}} \mathcal{P}(s' \mid s, a) V^*(s') \right]
$$


## 5. The Banach Fixed Point Theorem & Contraction Operators

Consider the vector space $\mathbb{R}^{|\mathcal{S}|}$ equipped with the **infinity norm**:

$$
\|V\|_\infty \equiv \max_{s \in \mathcal{S}} |V(s)|
$$

Under this metric, $(\mathbb{R}^{|\mathcal{S}|}, \|\cdot\|_\infty)$ forms a complete metric space (a Banach space). We define the **Bellman Optimality Operator** $T^*: \mathbb{R}^{|\mathcal{S}|} \to \mathbb{R}^{|\mathcal{S}|}$ as:

$$
(T^* V)(s) \equiv \max_{a \in \mathcal{A}} \left[ \mathcal{R}(s, a) + \gamma \sum_{s' \in \mathcal{S}} \mathcal{P}(s' \mid s, a) V(s') \right]
$$

In operator notation, the Bellman optimality equation is the fixed-point equation $T^* V^* = V^*$.

### Theorem: $T^*$ is a $\gamma$-Contraction Mapping

**Theorem:** For any two value vectors $U, V \in \mathbb{R}^{|\mathcal{S}|}$, the Bellman operator $T^*$ satisfies:

$$
\|T^* U - T^* V\|_\infty \le \gamma \|U - V\|_\infty
$$

#### Proof
Let $s \in \mathcal{S}$ be an arbitrary state. Suppose without loss of generality that $(T^* U)(s) \ge (T^* V)(s)$. Let $a^*_U = \arg\max_{a \in \mathcal{A}} [\mathcal{R}(s, a) + \gamma \sum_{s'} \mathcal{P}(s' \mid s, a) U(s')]$. Then:

$$
\begin{aligned}
(T^* U)(s) - (T^* V)(s) &= \left[ \mathcal{R}(s, a^*_U) + \gamma \sum_{s' \in \mathcal{S}} \mathcal{P}(s' \mid s, a^*_U) U(s') \right] - \max_{a \in \mathcal{A}} \left[ \mathcal{R}(s, a) + \gamma \sum_{s' \in \mathcal{S}} \mathcal{P}(s' \mid s, a) V(s') \right] \\
&\le \left[ \mathcal{R}(s, a^*_U) + \gamma \sum_{s' \in \mathcal{S}} \mathcal{P}(s' \mid s, a^*_U) U(s') \right] - \left[ \mathcal{R}(s, a^*_U) + \gamma \sum_{s' \in \mathcal{S}} \mathcal{P}(s' \mid s, a^*_U) V(s') \right] \\
&= \gamma \sum_{s' \in \mathcal{S}} \mathcal{P}(s' \mid s, a^*_U) \left( U(s') - V(s') \right) \\
&\le \gamma \sum_{s' \in \mathcal{S}} \mathcal{P}(s' \mid s, a^*_U) \|U - V\|_\infty = \gamma \|U - V\|_\infty
\end{aligned}
$$

Reversing the roles of $U$ and $V$ when $(T^* V)(s) > (T^* U)(s)$ yields $(T^* V)(s) - (T^* U)(s) \le \gamma \|U - V\|_\infty$. Hence $|(T^* U)(s) - (T^* V)(s)| \le \gamma \|U - V\|_\infty$ for all $s \in \mathcal{S}$. Taking the maximum over $s \in \mathcal{S}$ completes the proof:

$$
\|T^* U - T^* V\|_\infty \le \gamma \|U - V\|_\infty \quad \blacksquare
$$

### Convergence Guarantees via Banach's Fixed-Point Theorem

Because $T^*$ is a contraction mapping on a complete metric space:
1. **Uniqueness:** There exists a unique fixed point $V^* \in \mathbb{R}^{|\mathcal{S}|}$ satisfying $T^* V^* = V^*$.
2. **Global Convergence:** For any initial vector $V_0$, the sequence $V_{k+1} = T^* V_k$ converges to $V^*$: $\lim_{k \to \infty} V_k = V^*$.
3. **Geometric Convergence Rate (A Priori Bound):**

$$
\|V_k - V^*\|_\infty \le \frac{\gamma^k}{1 - \gamma} \|V_1 - V_0\|_\infty
$$

4. **A Posteriori Error Bound:** At iteration $k$:

$$
\|V_k - V^*\|_\infty \le \frac{\gamma}{1 - \gamma} \|V_k - V_{k-1}\|_\infty
$$


## 6. Dynamic Programming Algorithms: Value Iteration & Policy Iteration

Equipped with the contraction property, we now formalize **Value Iteration** and **Policy Iteration**.

---

### 6.1 Value Iteration (VI)

Value Iteration applies successive approximations directly to the Bellman optimality operator.

#### Algorithm Specification
1. Initialize $V_0(s) = 0$ for all $s \in \mathcal{S}$.
2. For iteration $k = 0, 1, 2, \dots$:
   - For each state $s \in \mathcal{S}$, update:

$$
V_{k+1}(s) \leftarrow \max_{a \in \mathcal{A}} \left[ \mathcal{R}(s, a) + \gamma \sum_{s' \in \mathcal{S}} \mathcal{P}(s' \mid s, a) V_k(s') \right]
$$

   - Compute error: $\Delta_k = \|V_{k+1} - V_k\|_\infty = \max_{s \in \mathcal{S}} |V_{k+1}(s) - V_k(s)|$.
   - **Stopping Condition:** If $\Delta_k < \epsilon \frac{1 - \gamma}{2\gamma}$, stop and extract greedy policy:

$$
\pi^*(s) = \arg\max_{a \in \mathcal{A}} \left[ \mathcal{R}(s, a) + \gamma \sum_{s' \in \mathcal{S}} \mathcal{P}(s' \mid s, a) V_{k+1}(s') \right]
$$

#### $\epsilon$-Optimality Guarantee

If $\|V_{k+1} - V_k\|_\infty < \epsilon \frac{1 - \gamma}{2\gamma}$, then $\|V_{k+1} - V^*\|_\infty < \frac{\epsilon}{2}$. Let $\pi$ be the greedy policy extracted with respect to $V_{k+1}$. Then it is guaranteed that $\|V^\pi - V^*\|_\infty \le \frac{2\gamma}{1 - \gamma} \|V_{k+1} - V_k\|_\infty < \epsilon$.

- Per-iteration cost: $\mathcal{O}(|\mathcal{S}|^2 |\mathcal{A}|)$.
- Iterations to reach $\epsilon$: $\mathcal{O}\left( \frac{1}{1 - \gamma} \ln \frac{1}{\epsilon} \right)$.

---

### 6.2 Policy Iteration (PI)

Policy Iteration operates directly in policy space, alternating between evaluation and improvement.

#### Algorithm Specification
1. **Initialization:** Select an initial deterministic policy $\pi_0$.
2. For iteration $k = 0, 1, 2, \dots$:
   - **Step 1: Policy Evaluation.** Solve the exact linear system for $\mathbf{V}^{\pi_k}$:

$$
(I - \gamma \mathbf{P}_{\pi_k}) \mathbf{V}^{\pi_k} = \mathbf{R}_{\pi_k}
$$

   - **Step 2: Policy Improvement.** Greedily update the policy:

$$
\pi_{k+1}(s) = \arg\max_{a \in \mathcal{A}} \left[ \mathcal{R}(s, a) + \gamma \sum_{s' \in \mathcal{S}} \mathcal{P}(s' \mid s, a) V^{\pi_k}(s') \right], \quad \forall s \in \mathcal{S}
$$

   - **Termination Criterion:** If $\pi_{k+1}(s) = \pi_k(s)$ for all $s \in \mathcal{S}$, terminate with optimal policy $\pi^* = \pi_k$.

#### Policy Improvement Theorem & Finite Convergence

**Theorem:** If $\pi'$ satisfies $Q^\pi(s, \pi'(s)) \ge V^\pi(s)$ for all $s \in \mathcal{S}$, then $V^{\pi'}(s) \ge V^\pi(s)$ for all $s \in \mathcal{S}$.

*Proof Sketch:* Unrolling $V^\pi(s) \le \mathcal{R}(s, \pi'(s)) + \gamma \sum_{s'} \mathcal{P}(s' \mid s, \pi'(s)) V^\pi(s')$ indefinitely yields $V^\pi(s) \le \mathbb{E}_{\pi'} [\sum_{k=0}^\infty \gamma^k R_{t+k+1} \mid S_t = s] = V^{\pi'}(s)$. $\blacksquare$

Because there are only $|\mathcal{A}|^{|\mathcal{S}|}$ distinct deterministic policies and each non-terminal iteration strictly improves the value function for at least one state, **Policy Iteration terminates in at most $|\mathcal{A}|^{|\mathcal{S}|}$ iterations**. In practice, it converges in 3–15 iterations.


## 7. Discounted Horizons vs. Average-Cost MDPs

In long-running cloud infrastructure systems, worker pools operate 24 hours a day without an explicit termination date. This introduces the **Average-Cost (or Long-Run Gain) MDP**.

### The Average-Cost Formulation

The long-run average reward (gain) $g^\pi(s)$ of a policy $\pi$ from initial state $s$ is defined as:

$$
g^\pi(s) \equiv \lim_{T \to \infty} \frac{1}{T} \mathbb{E}_\pi \left[ \sum_{t=0}^{T-1} R_{t+1} \;\middle|\; S_0 = s \right]
$$

### Unichain vs. Multichain Structure
- **Unichain MDP:** For every policy $\pi$, $\mathbf{P}_\pi$ induces a Markov chain consisting of a single recurrent class plus optional transient states. The stationary distribution $\boldsymbol{\mu}^\pi$ is unique, making the gain **independent of the starting state**:

$$
g^\pi = \sum_{s \in \mathcal{S}} \mu^\pi(s) \mathcal{R}(s, \pi(s)), \quad \forall s \in \mathcal{S}
$$

- **Multichain MDP:** Policies induce multiple disjoint recurrent classes, meaning $g^\pi(s)$ depends on the initial state $s$.

### The Average-Cost Bellman Equation (Poisson Equation)

Under unichain conditions, we define the **relative value function** $h^*(s) \equiv \lim_{\gamma \to 1^-} (V_\gamma^*(s) - V_\gamma^*(s_0))$, which satisfies:

$$
g^* + h^*(s) = \max_{a \in \mathcal{A}} \left[ \mathcal{R}(s, a) + \sum_{s' \in \mathcal{S}} \mathcal{P}(s' \mid s, a) h^*(s') \right], \quad \forall s \in \mathcal{S}
$$

where $g^*$ is the optimal scalar gain and $h^*(s_0) = 0$ at a reference state $s_0$.

### Systems Comparison

| Criterion | Systems Engineering Application | Primary Rationale |
| :--- | :--- | :--- |
| **Discounted ($\gamma < 1$)** | Ephemeral CI test runs, spot worker nodes, canary rollbacks | Accounts for node preemption risk ($1 - \gamma$), values immediate latency recovery over distant future throughput |
| **Average Cost ($g^*$)** | 24/7 Dedicated hardware testbeds, bare-metal server maintenance | Evaluates continuous steady-state throughput and amortized dollar costs without artificial time discounting |


## 8. Complete Step-by-Step Worked Numerical Example

We now work through a concrete numerical example with real numbers: computing 3 manual iterations of Value Iteration, solving the exact Policy Iteration linear systems, and demonstrating convergence to the optimal maintenance policy $\pi^* = [R, S, S]^T$.

---

### Systems Scenario: Hardware Worker Node Maintenance

An automated hardware testbed runs continuous integration test suites on runner nodes that degrade under thermal stress and memory leaks.

#### State Space $\mathcal{S}$
- **State 0: Healthy ($H$)** — Pristine condition, caches warm, 0% throttling.
- **State 1: Degraded ($D$)** — Memory fragmentation, elevated latency, frequent job timeouts.
- **State 2: Failed ($F$)** — Kernel panic, zombie process, zero job execution.

$$
\mathcal{S} = \{0: H, \; 1: D, \; 2: F\}
$$

#### Action Space $\mathcal{A}$
- **Action 0: Run ($R$)** — Accept incoming test jobs.
- **Action 1: Service ($S$)** — Take offline, flush caches, reboot, and power-cycle.

$$
\mathcal{A} = \{0: R, \; 1: S\}
$$

#### Transition Probability Matrices

$$
\mathbf{P}_R = \begin{bmatrix}
0.70 & 0.25 & 0.05 \\
0.00 & 0.50 & 0.50 \\
0.00 & 0.00 & 1.00
\end{bmatrix}, \quad \mathbf{P}_S = \begin{bmatrix}
1.00 & 0.00 & 0.00 \\
0.90 & 0.10 & 0.00 \\
0.80 & 0.20 & 0.00
\end{bmatrix}
$$

- Under $R$, a healthy node stays healthy with 70%, degrades with 25%, and crashes with 5%. A degraded node has a 50% crash chance. A failed node remains dead with 100%.
- Under $S$, servicing restores degraded nodes to healthy with 90% (10% residual defect), and failed nodes to healthy with 80% (20% degraded).

#### Immediate Rewards and Discount Factor
- $\mathbf{R}_R = [10.0, \; 4.0, \; -20.0]^T$ ($+10$ throughput in $H$, $+4$ throttled in $D$, $-20$ outage penalty in $F$).
- $\mathbf{R}_S = [0.0, \; -5.0, \; -5.0]^T$ ($0$ in $H$, $-5$ diagnostic/reboot overhead in $D$ and $F$).
- Discount factor: **$\gamma = 0.90$** ($H_{\text{eff}} = \frac{1}{1 - 0.9} = 10$ steps).


### 8.1 Manual Value Iteration (Three Full Iterations)

Initialize $V_0 = [0.0, \; 0.0, \; 0.0]^T$.

---

#### Iteration $k = 1$
Compute $Q_1(s, a) = \mathcal{R}(s, a) + 0.90 \sum_{s'} \mathcal{P}(s' \mid s, a) V_0(s')$:

$$
\begin{aligned}
Q_1(\cdot, R) &= \mathbf{R}_R + 0.90 \mathbf{P}_R V_0 = [10.0, \; 4.0, \; -20.0]^T \\
Q_1(\cdot, S) &= \mathbf{R}_S + 0.90 \mathbf{P}_S V_0 = [0.0, \; -5.0, \; -5.0]^T
\end{aligned}
$$

Maximizing $V_1(s) = \max\{Q_1(s, R), Q_1(s, S)\}$:
- $s = H: \max\{10.0, 0.0\} = 10.0 \implies \pi_1(H) = R$
- $s = D: \max\{4.0, -5.0\} = 4.0 \implies \pi_1(D) = R$
- $s = F: \max\{-20.0, -5.0\} = -5.0 \implies \pi_1(F) = S$

$$
V_1 = \begin{bmatrix} 10.0 \\ 4.0 \\ -5.0 \end{bmatrix}, \quad \pi_1 = \begin{bmatrix} R \\ R \\ S \end{bmatrix}, \quad \|V_1 - V_0\|_\infty = 10.0
$$

---

#### Iteration $k = 2$
Compute expected future values under both actions:

$$
\mathbf{P}_R V_1 = \begin{bmatrix}
0.70(10.0) + 0.25(4.0) + 0.05(-5.0) \\
0.00(10.0) + 0.50(4.0) + 0.50(-5.0) \\
0.00(10.0) + 0.00(4.0) + 1.00(-5.0)
\end{bmatrix} = \begin{bmatrix} 7.0 + 1.0 - 0.25 \\ 2.0 - 2.5 \\ -5.0 \end{bmatrix} = \begin{bmatrix} 7.75 \\ -0.50 \\ -5.00 \end{bmatrix}
$$

$$
Q_2(\cdot, R) = \begin{bmatrix} 10.0 + 0.90(7.75) \\ 4.0 + 0.90(-0.50) \\ -20.0 + 0.90(-5.00) \end{bmatrix} = \begin{bmatrix} 16.975 \\ 3.550 \\ -24.500 \end{bmatrix}
$$

$$
\mathbf{P}_S V_1 = \begin{bmatrix}
1.00(10.0) \\
0.90(10.0) + 0.10(4.0) \\
0.80(10.0) + 0.20(4.0)
\end{bmatrix} = \begin{bmatrix} 10.00 \\ 9.40 \\ 8.80 \end{bmatrix} \implies Q_2(\cdot, S) = \begin{bmatrix} 0.0 + 0.90(10.00) \\ -5.0 + 0.90(9.40) \\ -5.0 + 0.90(8.80) \end{bmatrix} = \begin{bmatrix} 9.000 \\ 3.460 \\ 2.920 \end{bmatrix}
$$

Maximizing $V_2(s) = \max\{Q_2(s, R), Q_2(s, S)\}$:
- $s = H: \max\{16.975, 9.000\} = 16.975 \implies \pi_2(H) = R$
- $s = D: \max\{3.550, 3.460\} = 3.550 \implies \pi_2(D) = R$ (running edges out servicing by $0.090$)
- $s = F: \max\{-24.500, 2.920\} = 2.920 \implies \pi_2(F) = S$

$$
V_2 = \begin{bmatrix} 16.975 \\ 3.550 \\ 2.920 \end{bmatrix}, \quad \pi_2 = \begin{bmatrix} R \\ R \\ S \end{bmatrix}, \quad \|V_2 - V_1\|_\infty = \max\{6.975, 0.450, 7.920\} = 7.920
$$

---

#### Iteration $k = 3$
Compute expected future values using $V_2$:

$$
\mathbf{P}_R V_2 = \begin{bmatrix} 0.70(16.975) + 0.25(3.550) + 0.05(2.920) \\ 0.50(3.550) + 0.50(2.920) \\ 1.00(2.920) \end{bmatrix} = \begin{bmatrix} 12.9160 \\ 3.2350 \\ 2.9200 \end{bmatrix} \implies Q_3(\cdot, R) = \begin{bmatrix} 10.0 + 0.90(12.9160) \\ 4.0 + 0.90(3.2350) \\ -20.0 + 0.90(2.9200) \end{bmatrix} = \begin{bmatrix} 21.6244 \\ 6.9115 \\ -17.3720 \end{bmatrix}
$$

$$
\mathbf{P}_S V_2 = \begin{bmatrix} 1.00(16.975) \\ 0.90(16.975) + 0.10(3.550) \\ 0.80(16.975) + 0.20(3.550) \end{bmatrix} = \begin{bmatrix} 16.9750 \\ 15.6325 \\ 14.2900 \end{bmatrix} \implies Q_3(\cdot, S) = \begin{bmatrix} 0.0 + 0.90(16.9750) \\ -5.0 + 0.90(15.6325) \\ -5.0 + 0.90(14.2900) \end{bmatrix} = \begin{bmatrix} 15.2775 \\ 9.0693 \\ 7.8610 \end{bmatrix}
$$

Maximizing $V_3(s) = \max\{Q_3(s, R), Q_3(s, S)\}$:
- $s = H: \max\{21.6244, 15.2775\} = 21.6244 \implies \pi_3(H) = R$
- $s = D: \max\{6.9115, 9.0693\} = 9.0693 \implies \mathbf{\pi_3(D) = S}$! *(Policy switches from $R$ to $S$)*
- $s = F: \max\{-17.3720, 7.8610\} = 7.8610 \implies \pi_3(F) = S$

$$
V_3 = \begin{bmatrix} 21.6244 \\ 9.0693 \\ 7.8610 \end{bmatrix}, \quad \pi_3 = \begin{bmatrix} R \\ S \\ S \end{bmatrix}
$$

At iteration 3, the future cost of catastrophic failure in state $F$ has propagated backward: the algorithm realizes that servicing a degraded runner ($Q=9.0693$) is far better than running it ($Q=6.9115$). The policy has converged to $\mathbf{\pi^* = [R, S, S]^T}$.


### 8.2 Exact Policy Iteration (Full Execution)

We now solve the exact Policy Iteration linear systems to witness finite convergence.

---

#### Step 1: Initial Policy Evaluation ($\pi_0 = [R, R, S]^T$)
The induced transition matrix and reward vector for $\pi_0 = [R, R, S]^T$ are:

$$
\mathbf{P}_{\pi_0} = \begin{bmatrix}
0.70 & 0.25 & 0.05 \\
0.00 & 0.50 & 0.50 \\
0.80 & 0.20 & 0.00
\end{bmatrix}, \quad \mathbf{R}_{\pi_0} = \begin{bmatrix} 10.0 \\ 4.0 \\ -5.0 \end{bmatrix}
$$

We solve the linear system $(I - 0.90 \mathbf{P}_{\pi_0}) \mathbf{V}^{\pi_0} = \mathbf{R}_{\pi_0}$:

$$
\begin{bmatrix}
0.370 & -0.225 & -0.045 \\
0.000 & 0.550 & -0.450 \\
-0.720 & -0.180 & 1.000
\end{bmatrix} \mathbf{V}^{\pi_0} = \begin{bmatrix} 10.0 \\ 4.0 \\ -5.0 \end{bmatrix} \implies \mathbf{V}^{\pi_0} = \begin{bmatrix} 60.2874 \\ 45.3798 \\ 46.5753 \end{bmatrix}
$$

#### Step 2: Policy Improvement on $\pi_0$
Evaluate Q-values under both actions using $\mathbf{V}^{\pi_0}$:

$$
\begin{aligned}
Q^{\pi_0}(\cdot, R) &= \mathbf{R}_R + 0.90 \mathbf{P}_R \mathbf{V}^{\pi_0} = \begin{bmatrix} 10.0 + 0.90(55.8749) \\ 4.0 + 0.90(45.9776) \\ -20.0 + 0.90(46.5753) \end{bmatrix} = \begin{bmatrix} 60.2874 \\ 45.3798 \\ 21.9178 \end{bmatrix} \\
Q^{\pi_0}(\cdot, S) &= \mathbf{R}_S + 0.90 \mathbf{P}_S \mathbf{V}^{\pi_0} = \begin{bmatrix} 0.0 + 0.90(60.2874) \\ -5.0 + 0.90(58.7966) \\ -5.0 + 0.90(57.3059) \end{bmatrix} = \begin{bmatrix} 54.2587 \\ \mathbf{47.9170} \\ 46.5753 \end{bmatrix}
\end{aligned}
$$

Selecting greedy actions:
- In $H$: $60.2874 > 54.2587 \implies R$
- In $D$: $\mathbf{47.9170 > 45.3798 \implies S}$ *(Strict improvement!)*
- In $F$: $46.5753 > 21.9178 \implies S$

The policy strictly improves to $\pi_1 = [R, S, S]^T$.

---

#### Step 3: Second Policy Evaluation ($\pi_1 = [R, S, S]^T$)
The induced transition matrix and reward vector for $\pi_1$ are:

$$
\mathbf{P}_{\pi_1} = \begin{bmatrix}
0.70 & 0.25 & 0.05 \\
0.90 & 0.10 & 0.00 \\
0.80 & 0.20 & 0.00
\end{bmatrix}, \quad \mathbf{R}_{\pi_1} = \begin{bmatrix} 10.0 \\ -5.0 \\ -5.0 \end{bmatrix}
$$

Solving $(I - 0.90 \mathbf{P}_{\pi_1}) \mathbf{V}^{\pi_1} = \mathbf{R}_{\pi_1}$:

$$
\begin{bmatrix}
0.370 & -0.225 & -0.045 \\
-0.810 & 0.910 & 0.000 \\
-0.720 & -0.180 & 1.000
\end{bmatrix} \mathbf{V}^{\pi_1} = \begin{bmatrix} 10.0 \\ -5.0 \\ -5.0 \end{bmatrix} \implies \mathbf{V}^{\pi_1} = \begin{bmatrix} 65.2823 \\ 52.6139 \\ 51.4738 \end{bmatrix}
$$

Notice that $V^{\pi_1}(s) > V^{\pi_0}(s)$ strictly across every single state ($65.28 > 60.29$, $52.61 > 45.38$, $51.47 > 46.58$).

#### Step 4: Second Policy Improvement on $\pi_1$
Compute Q-values using $\mathbf{V}^{\pi_1}$:

$$
\begin{aligned}
Q^{\pi_1}(\cdot, R) &= \mathbf{R}_R + 0.90 \mathbf{P}_R \mathbf{V}^{\pi_1} = [65.2823, \; 50.8394, \; 26.3264]^T \\
Q^{\pi_1}(\cdot, S) &= \mathbf{R}_S + 0.90 \mathbf{P}_S \mathbf{V}^{\pi_1} = [58.7541, \; 52.6139, \; 51.4738]^T
\end{aligned}
$$

Selecting greedy actions:
- In $H$: $\max\{65.2823, 58.7541\} = 65.2823 \implies \mathbf{R}$
- In $D$: $\max\{50.8394, 52.6139\} = 52.6139 \implies \mathbf{S}$
- In $F$: $\max\{26.3264, 51.4738\} = 51.4738 \implies \mathbf{S}$

Thus $\pi_2 = [R, S, S]^T = \pi_1$. 

**Termination Reached!** Because $\pi_2 = \pi_1$, Policy Iteration terminates in exactly **2 iterations**, proving that $\mathbf{\pi^* = [R, S, S]^T}$ is mathematically optimal with value vector $\mathbf{V}^* = [65.2823, \; 52.6139, \; 51.4738]^T$.


## 9. Summary & Bridge to Partially Observable Systems

In this chapter, we transitioned from static combinatorial selection to dynamic sequential control:
1. **The Markov Property:** States must be sufficient statistics of history to render future trajectories conditionally independent of the past.
2. **Bellman Optimality:** Formulates optimal control recursively, linking immediate reward to expected discounted downstream return.
3. **Contraction Mappings:** The Bellman operator $T^*$ is an $\ell_\infty$ contraction with modulus $\gamma$. By the Banach Fixed Point Theorem, Value Iteration is guaranteed to converge geometrically to the unique global optimum from any initialization.
4. **Policy Iteration:** Combines exact linear system evaluation $(I - \gamma \mathbf{P}_\pi)^{-1} \mathbf{R}_\pi$ with greedy improvement, achieving exact finite convergence in remarkably few iterations.

### The Looming Challenge: Hidden States

Throughout this chapter, we operated under a critical assumption: **perfect state observability**. We assumed that the fleet controller always knows with 100% certainty whether a hardware runner node is Healthy, Degraded, or Failed.

In production environments, **physical states are rarely directly observable**:
- Silent memory corruption, race conditions, and thermal throttling do not announce themselves.
- Flaky tests produce false alarms (failing on healthy nodes) and false negatives (passing on degraded nodes).
- Automated health telemetry probes are noisy, delayed, and imperfect.

When physical state $S_t$ is hidden, how can an automated controller make optimal decisions based solely on noisy observation histories? In **[Chapter 05: Partially Observable Markov Decision Processes (POMDPs) & Belief States](05_pomdps_and_belief_state_mdps.ipynb)**, we generalize the Bellman framework to continuous belief simplexes $\mathbf{b}_t \in \Delta^{|\mathcal{S}|}$, proving that optimal value functions in partially observable systems are piecewise linear and convex (PWLC).

---
**[Course Home](../README.md) | [Python Lab](../python/04_markov_decision_processes_and_bellman_optimality.ipynb) | [Prev: Ch 03: Greedy Bounds & CELF](03_nemhauser_wolsey_greedy_guarantees.ipynb) | [Next: Ch 05: POMDPs & Belief States](05_pomdps_and_belief_state_mdps.ipynb)**
